In [12]:
import yfinance as yf
import pandas as pd
pd.options.display.float_format = '{:,.2f}'.format

ticker = "2454.TW"
stock = yf.Ticker(ticker)

# 獲取三大報表 (DataFrame 格式)
income_stmt = stock.financials        # 損益表
balance_sheet = stock.balance_sheet   # 資產負債表
cash_flow = stock.cashflow            # 年度現金流量表
quart_cash_flow = stock.quarterly_cash_flow #季度現金流
info = stock.info

In [ ]:
def calculate_market_wacc(ticker_symbol, rf=0.04, erp=0.05, rd=0.05):
    """
    rf: 無風險利率 (2026年假設 4%)
    erp: 股權風險溢價 (假設 5%)
    rd: 稅前債務成本 (假設 5%)
    """
    stock = yf.Ticker(ticker_symbol)
    info = stock.info

    # 取得季度資產負債表
    bs = stock.quarterly_balance_sheet
    
    # 股權成本 Re
    beta = info.get('beta', 1.0)
    re = rf + (beta * erp)
    
    # 權重計算
    market_cap = info.get('marketCap')
    total_debt = bs.loc['Total Debt'].iloc[0] if 'Total Debt' in bs.index else 0
    v = market_cap + total_debt
    
    w_e = market_cap / v
    w_d = total_debt / v
    tax_rate = 0.20
    
    wacc = (w_e * re) + (w_d * rd * (1 - tax_rate))
    return wacc

def get_dcf_valuation(ticker_symbol, ex_wacc=None , growth_rate=0.10, terminal_growth=0.02):
    
    stock = yf.Ticker(ticker_symbol)
    info = stock.info
    # 取得季度資產負債表
    bs = stock.quarterly_balance_sheet
    
    # 1. 取得 TTM 自由現金流 (加總最近四季)
    q_cf = stock.quarterly_cashflow
    if 'Free Cash Flow' in q_cf.index:
        ttm_fcf = q_cf.loc['Free Cash Flow'][:4].sum()
    else:
        # 備用方案：營業現金流 - 資本支出
        ocf = q_cf.loc['Operating Cash Flow'][:4].sum()
        capex = abs(q_cf.loc['Capital Expenditure'][:4].sum())
        ttm_fcf = ocf - capex
    
    if ttm_fcf <= 0:
        return {"錯誤": "自由現金流為負，不適用DCF估值"}

    # 2. 計算WACC
    wacc = ex_wacc if ex_wacc else calculate_market_wacc(ticker_symbol)
    if wacc <= terminal_growth:
        raise ValueError(f"WACC ({wacc:.2%}) 必須大於終端增長率 ({terminal_growth:.2%})")

    # 3. 計算未來 5 年現金流折現 (PV)
    # 公式：$$PV = \sum_{t=1}^{n} \frac{FCF_t}{(1+WACC)^t}$$
    future_fcf = []
    pv_fcf = []
    current_fcf = ttm_fcf
    
    for i in range(1, 6):
        current_fcf *= (1 + growth_rate)
        discounted = current_fcf / ((1 + wacc) ** i)
        pv_fcf.append(discounted)
    
    # 4. 計算終值 (Terminal Value) 並折現
    # 公式：$$TV = \frac{FCF_n \times (1+g_n)}{WACC - g_n}$$
    last_fcf = current_fcf
    tv = (last_fcf * (1 + terminal_growth)) / (wacc - terminal_growth)
    pv_tv = tv / ((1 + wacc) ** 5)
    
    # 5. 計算每股內在價值
    enterprise_value = sum(pv_fcf) + pv_tv
    cash = bs.loc['Cash And Cash Equivalents'].iloc[0] if 'Cash And Cash Equivalents' in bs.index else 0
    short_term_invest = bs.loc['Other Short Term Investments'].iloc[0] if 'Other Short Term Investments' in bs.index else 0
    total_cash = cash + short_term_invest
    total_debt = bs.loc['Total Debt'].iloc[0] if 'Total Debt' in bs.index else 0
    equity_value = enterprise_value + total_cash - total_debt
    shares_outstanding = info.get('sharesOutstanding')
    intrinsic_value_per_share = equity_value / shares_outstanding
    tv_percentage = pv_tv / enterprise_value
    # 6. 結果比較
    current_price = info.get('currentPrice')
    margin_of_safety = (intrinsic_value_per_share - current_price) / intrinsic_value_per_share
    
    # 定義波動性：Beta 大於 1.2 視為高波動股票
    beta = info.get('beta', 1.0)
    is_volatile = beta > 1.2

    # 動態設定安全邊際門檻
    # 高波動(如科技股)要求 30% 折扣；低波動(如權值、公用事業)要求 15% 折扣
    target_mos = 0.30 if is_volatile else 0.15

    # 最終判斷
    if margin_of_safety > target_mos:
        result = "便宜"
    elif margin_of_safety > 0:
        result = "合理"
    else:
        result = "昂貴"

    return {
        "股票代碼": ticker_symbol,
        "目前股價": f"{current_price:.2f}",
        "WACC": f"{wacc:.2%}",
        "安全邊際": f"{margin_of_safety:.2%}",
        "beta": f"{beta}",
        "建議": f"{result}",
        # --- 詳細參數區 ---
        "企業價值(EV)": f"{enterprise_value:,.0f}",
        "終值佔比": f"{tv_percentage:.2%}",
        "總現金": f"{total_cash:,.0f}",
        "總負債": f"{total_debt:,.0f}",
        "內在價值": f"{intrinsic_value_per_share:.2f}",
        "股權價值": f"{equity_value:,.0f}",
        "流通股數": f"{shares_outstanding:,.0f}"
    }

In [47]:
result = get_dcf_valuation("2454.TW", growth_rate=0.15)
print(pd.Series(result))

股票代碼                  2454.TW
目前股價                  1845.00
WACC                    9.06%
安全邊際                    5.20%
beta                    1.021
建議                         合理
企業價值(EV)    2,923,006,875,781
終值佔比                   76.22%
總現金           210,603,730,000
總負債            27,160,469,000
內在價值                  1946.26
股權價值        3,106,450,136,781
流通股數            1,596,110,205
dtype: object


In [50]:
wacc = calculate_market_wacc("2330.TW", rf=0.04, erp=0.07, rd=0.05)
result = get_dcf_valuation("2330.TW", ex_wacc=0.08, growth_rate=0.25)
print(pd.Series(result))

股票代碼                   2330.TW
目前股價                   1880.00
WACC                     8.00%
安全邊際                   -21.73%
beta                     1.272
建議                          昂貴
企業價值(EV)    38,192,989,822,745
終值佔比                    81.68%
總現金          2,805,054,104,000
總負債            949,207,252,000
內在價值                   1544.35
股權價值        40,048,836,674,745
流通股數            25,932,524,521
dtype: object


In [31]:
result = get_dcf_valuation("2535.TW", growth_rate=0.02)
print(pd.Series(result))

股票代碼               2535.TW
目前股價                 70.70
WACC                 4.86%
安全邊際                78.66%
beta                 0.258
建議                      便宜
企業價值(EV)    87,265,261,969
總現金          8,161,356,000
總負債          9,322,066,000
內在價值                331.31
股權價值        86,104,551,969
流通股數           259,891,739
dtype: object


In [42]:
result = get_dcf_valuation("2367.TW", growth_rate=0.02)
print(pd.Series(result))

股票代碼              2367.TW
目前股價                51.60
WACC                9.02%
安全邊際             -779.05%
beta                1.183
建議                     昂貴
企業價值(EV)    9,915,583,607
終值佔比               71.70%
總現金           746,297,000
總負債         6,516,215,000
內在價值                 5.87
股權價值        4,145,665,607
流通股數          706,253,175
dtype: object
